# Milestone 4.34 - Missing Value Handling Strategies

This notebook demonstrates comprehensive handling of missing values in Pandas DataFrames:

- Load a DataFrame with missing values
- Apply drop strategies where appropriate
- Apply fill strategies where appropriate
- Compare the impact of each approach
- Clean, readable, and well-organized code

**Note**: No modeling, visualization, or advanced transformations are performed.

In [ ]:
import pandas as pd
import numpy as np

# Create a healthcare dataset with intentional missing values
np.random.seed(456)  # For reproducible results

# Generate base patient data
num_patients = 100
patient_ids = range(3001, 3001 + num_patients)

# Create base data
data = {
    'patient_id': patient_ids,
    'name': [f'Patient_{i}' for i in range(1, num_patients + 1)],
    'age': np.random.randint(18, 85, num_patients),
    'gender': np.random.choice(['Male', 'Female'], num_patients),
    'blood_pressure_systolic': np.random.randint(90, 180, num_patients),
    'blood_pressure_diastolic': np.random.randint(60, 120, num_patients),
    'heart_rate': np.random.randint(50, 120, num_patients),
    'temperature': np.round(np.random.uniform(96.0, 101.0, num_patients), 1),
    'department': np.random.choice(['Cardiology', 'Neurology', 'Orthopedics', 'Emergency', 'Pediatrics'], num_patients),
    'admission_date': pd.date_range('2024-01-01', periods=num_patients, freq='3H'),
    'treatment': np.random.choice(['Surgery', 'Medication', 'Therapy', 'Observation'], num_patients),
    'insurance_type': np.random.choice(['Private', 'Medicare', 'Medicaid', 'Self-Pay'], num_patients),
    'total_cost': np.round(np.random.uniform(1000, 50000, num_patients), 2),
    'length_of_stay': np.random.randint(1, 15, num_patients)
}

# Create DataFrame
df = pd.DataFrame(data)

# Intentionally introduce missing values to demonstrate handling
# Scenario 1: Missing vital signs (moderate missing)
missing_vitals_indices = np.random.choice(df.index, size=15, replace=False)
df.loc[missing_vitals_indices[:6], 'heart_rate'] = np.nan
df.loc[missing_vitals_indices[6:10], 'temperature'] = np.nan
df.loc[missing_vitals_indices[10:15], 'blood_pressure_systolic'] = np.nan

# Scenario 2: Missing administrative data (high missing)
missing_admin_indices = np.random.choice(df.index, size=20, replace=False)
df.loc[missing_admin_indices[:12], 'insurance_type'] = np.nan
df.loc[missing_admin_indices[12:20], 'treatment'] = np.nan

# Scenario 3: Missing cost data (low missing)
missing_cost_indices = np.random.choice(df.index, size=5, replace=False)
df.loc[missing_cost_indices, 'total_cost'] = np.nan

# Scenario 4: Multiple missing values in same rows
multi_missing_indices = np.random.choice(df.index, size=8, replace=False)
for idx in multi_missing_indices:
    df.loc[idx, ['blood_pressure_diastolic', 'department', 'length_of_stay']] = np.nan

print("Original DataFrame with missing values:")
print(f"Dataset shape: {df.shape}")
print(f"Total missing values: {df.isnull().sum().sum()}")
print(f"Missing percentage: {(df.isnull().sum().sum() / (df.shape[0] * df.shape[1])) * 100:.2f}%")

## 1. Initial Missing Value Detection

Detecting missing values before applying any handling strategies.

In [ ]:
# Initial missing value analysis
print("=== INITIAL MISSING VALUE ANALYSIS ===")

# Overall missing value statistics
total_missing = df.isnull().sum().sum()
total_cells = df.shape[0] * df.shape[1]
missing_percentage = (total_missing / total_cells) * 100

print(f"Original DataFrame shape: {df.shape}")
print(f"Total missing values: {total_missing}")
print(f"Missing percentage: {missing_percentage:.2f}%")

# Column-wise missing values
print("\nMissing values by column:")
missing_by_column = df.isnull().sum()
missing_percentage_by_column = (missing_by_column / len(df)) * 100

for col in df.columns:
    count = missing_by_column[col]
    percentage = missing_percentage_by_column[col]
    if count > 0:
        print(f"  {col}: {count} missing ({percentage:.1f}%)")

# Row-wise missing values
missing_by_row = df.isnull().sum(axis=1)
rows_with_missing = (missing_by_row > 0).sum()
print(f"\nRows with missing values: {rows_with_missing} ({(rows_with_missing/len(df))*100:.1f}%)")

# Store original shape for comparison
original_shape = df.shape
print(f"\nOriginal shape stored for comparison: {original_shape}")

## 2. Drop Strategies

Applying various drop strategies using `dropna()`.

In [ ]:
print("=== DROP STRATEGIES ===")

# Strategy 1: Drop all rows with any missing values
print("Strategy 1: Drop all rows with any missing values")
df_drop_any = df.dropna()
print(f"  Original shape: {original_shape}")
print(f"  After dropna(): {df_drop_any.shape}")
print(f"  Rows removed: {original_shape[0] - df_drop_any.shape[0]}")
print(f"  Data retained: {(df_drop_any.shape[0] / original_shape[0]) * 100:.1f}%")

# Strategy 2: Drop columns with any missing values
print("\nStrategy 2: Drop columns with any missing values")
df_drop_cols = df.dropna(axis=1)
print(f"  Original shape: {original_shape}")
print(f"  After drop columns: {df_drop_cols.shape}")
print(f"  Columns removed: {original_shape[1] - df_drop_cols.shape[1]}")
print(f"  Columns retained: {(df_drop_cols.shape[1] / original_shape[1]) * 100:.1f}%")
print(f"  Remaining columns: {list(df_drop_cols.columns)}")

# Strategy 3: Drop rows with more than 2 missing values
print("\nStrategy 3: Drop rows with more than 2 missing values")
df_drop_threshold = df.dropna(thresh=len(df.columns) - 2)  # Keep rows with at most 2 missing values
print(f"  Original shape: {original_shape}")
print(f"  After threshold drop: {df_drop_threshold.shape}")
print(f"  Rows removed: {original_shape[0] - df_drop_threshold.shape[0]}")
print(f"  Data retained: {(df_drop_threshold.shape[0] / original_shape[0]) * 100:.1f}%")

# Strategy 4: Drop specific columns with high missing rates
print("\nStrategy 4: Drop columns with >15% missing values")
missing_percentage = (df.isnull().sum() / len(df)) * 100
cols_to_drop = missing_percentage[missing_percentage > 15].index.tolist()
print(f"  Columns to drop: {cols_to_drop}")
df_drop_specific = df.drop(columns=cols_to_drop)
print(f"  Original shape: {original_shape}")
print(f"  After specific column drop: {df_drop_specific.shape}")
print(f"  Columns removed: {len(cols_to_drop)}")

# Strategy 5: Drop rows missing critical data
print("\nStrategy 5: Drop rows missing critical data (patient_id, age, department)")
critical_cols = ['patient_id', 'age', 'department']
df_drop_critical = df.dropna(subset=critical_cols)
print(f"  Original shape: {original_shape}")
print(f"  After critical data drop: {df_drop_critical.shape}")
print(f"  Rows removed: {original_shape[0] - df_drop_critical.shape[0]}")
print(f"  Data retained: {(df_drop_critical.shape[0] / original_shape[0]) * 100:.1f}%")

## 3. Fill Strategies

Applying various fill strategies using `fillna()`.

In [ ]:
print("=== FILL STRATEGIES ===")

# Strategy 1: Fill with constant values
print("Strategy 1: Fill with constant values")
df_fill_constant = df.copy()
df_fill_constant['heart_rate'] = df_fill_constant['heart_rate'].fillna(70)  # Normal heart rate
df_fill_constant['temperature'] = df_fill_constant['temperature'].fillna(98.6)  # Normal temperature
df_fill_constant['insurance_type'] = df_fill_constant['insurance_type'].fillna('Unknown')
df_fill_constant['treatment'] = df_fill_constant['treatment'].fillna('Pending')

print(f"  Original shape: {original_shape}")
print(f"  After constant fill: {df_fill_constant.shape}")
print(f"  Missing values remaining: {df_fill_constant.isnull().sum().sum()}")
print(f"  Reduction in missing: {total_missing - df_fill_constant.isnull().sum().sum()}")

# Strategy 2: Fill with mean for numeric columns
print("\nStrategy 2: Fill numeric columns with mean")
df_fill_mean = df.copy()
numeric_cols = df.select_dtypes(include=[np.number]).columns
for col in numeric_cols:
    if df_fill_mean[col].isnull().sum() > 0:
        mean_val = df_fill_mean[col].mean()
        df_fill_mean[col] = df_fill_mean[col].fillna(mean_val)
        print(f"    {col}: filled with mean {mean_val:.2f}")

print(f"  Missing values after mean fill: {df_fill_mean.isnull().sum().sum()}")

# Strategy 3: Fill with median for numeric columns
print("\nStrategy 3: Fill numeric columns with median")
df_fill_median = df.copy()
for col in numeric_cols:
    if df_fill_median[col].isnull().sum() > 0:
        median_val = df_fill_median[col].median()
        df_fill_median[col] = df_fill_median[col].fillna(median_val)
        print(f"    {col}: filled with median {median_val:.2f}")

print(f"  Missing values after median fill: {df_fill_median.isnull().sum().sum()}")

# Strategy 4: Fill with mode for categorical columns
print("\nStrategy 4: Fill categorical columns with mode")
df_fill_mode = df.copy()
categorical_cols = df.select_dtypes(include=['object']).columns
for col in categorical_cols:
    if df_fill_mode[col].isnull().sum() > 0:
        mode_val = df_fill_mode[col].mode()[0]
        df_fill_mode[col] = df_fill_mode[col].fillna(mode_val)
        print(f"    {col}: filled with mode '{mode_val}'")

print(f"  Missing values after mode fill: {df_fill_mode.isnull().sum().sum()}")

# Strategy 5: Fill with forward/backward fill for time series
print("\nStrategy 5: Fill with forward fill for time-based data")
df_fill_ffill = df.copy()
# Sort by admission_date for meaningful forward fill
df_fill_ffill = df_fill_ffill.sort_values('admission_date')
df_fill_ffill['length_of_stay'] = df_fill_ffill['length_of_stay'].ffill()
df_fill_ffill['length_of_stay'] = df_fill_ffill['length_of_stay'].bfill()  # Backward fill for remaining

print(f"  Missing values after forward fill: {df_fill_ffill.isnull().sum().sum()}")

# Strategy 6: Combined strategy
print("\nStrategy 6: Combined fill strategy")
df_fill_combined = df.copy()

# Fill numeric with median
for col in numeric_cols:
    if df_fill_combined[col].isnull().sum() > 0:
        df_fill_combined[col] = df_fill_combined[col].fillna(df_fill_combined[col].median())

# Fill categorical with mode
for col in categorical_cols:
    if df_fill_combined[col].isnull().sum() > 0:
        df_fill_combined[col] = df_fill_combined[col].fillna(df_fill_combined[col].mode()[0])

print(f"  Missing values after combined fill: {df_fill_combined.isnull().sum().sum()}")
print(f"  All missing values resolved: {df_fill_combined.isnull().sum().sum() == 0}")

## 4. Strategy Comparison

Comparing the impact of each approach on dataset shape and data quality.

In [ ]:
print("=== STRATEGY COMPARISON ===")

# Create comparison summary
strategies = {
    'Original': df,
    'Drop Any Missing': df_drop_any,
    'Drop Columns': df_drop_cols,
    'Drop Threshold': df_drop_threshold,
    'Drop Specific Cols': df_drop_specific,
    'Drop Critical': df_drop_critical,
    'Fill Constant': df_fill_constant,
    'Fill Mean': df_fill_mean,
    'Fill Median': df_fill_median,
    'Fill Mode': df_fill_mode,
    'Fill Forward': df_fill_ffill,
    'Fill Combined': df_fill_combined
}

comparison_data = []
for name, strategy_df in strategies.items():
    shape = strategy_df.shape
    missing_count = strategy_df.isnull().sum().sum()
    data_retention = (shape[0] / original_shape[0]) * 100
    column_retention = (shape[1] / original_shape[1]) * 100
    
    comparison_data.append({
        'Strategy': name,
        'Rows': shape[0],
        'Columns': shape[1],
        'Missing Values': missing_count,
        'Data Retention %': round(data_retention, 1),
        'Column Retention %': round(column_retention, 1)
    })

comparison_df = pd.DataFrame(comparison_data)
print("Strategy Comparison Summary:")
print(comparison_df.to_string(index=False))

# Detailed analysis
print("\n=== DETAILED ANALYSIS ===")

# Best strategies for different scenarios
print("\nBest strategies for different scenarios:")
print("  1. Maximum data retention (no rows dropped):")
max_retention = comparison_df[comparison_df['Data Retention %'] == 100]['Strategy'].tolist()
print(f"     {', '.join(max_retention)}")

print("\n  2. Complete missing value resolution:")
complete_resolution = comparison_df[comparison_df['Missing Values'] == 0]['Strategy'].tolist()
print(f"     {', '.join(complete_resolution)}")

print("\n  3. Balanced approach (good retention + resolution):")
balanced = comparison_df[(comparison_df['Data Retention %'] > 80) & 
                         (comparison_df['Missing Values'] < 10)]['Strategy'].tolist()
print(f"     {', '.join(balanced)}")

# Impact assessment
print("\n=== IMPACT ASSESSMENT ===")
print(f"Original dataset: {original_shape[0]} rows × {original_shape[1]} columns")
print(f"Total missing values: {total_missing}")
print(f"Missing percentage: {missing_percentage:.2f}%")

print("\nRecommended approach:")
if missing_percentage < 5:
    print("  Use combined fill strategy - preserves all data while resolving missing values")
elif missing_percentage < 15:
    print("  Use selective drop + fill - drop high-missing columns, fill the rest")
else:
    print("  Consider drop threshold strategy - remove rows with excessive missing data")

## 5. Decision Framework

Guidelines for choosing appropriate missing value handling strategies.

In [ ]:
print("=== DECISION FRAMEWORK FOR MISSING VALUE HANDLING ===")

# Analyze missing value patterns
print("\n1. MISSING VALUE PATTERN ANALYSIS:")
missing_by_col = df.isnull().sum()
high_missing_cols = missing_by_col[missing_by_col > len(df) * 0.2].index.tolist()
moderate_missing_cols = missing_by_col[(missing_by_col > len(df) * 0.05) & 
                                       (missing_by_col <= len(df) * 0.2)].index.tolist()
low_missing_cols = missing_by_col[missing_by_col <= len(df) * 0.05].index.tolist()

print(f"  High missing (>20%): {high_missing_cols}")
print(f"  Moderate missing (5-20%): {moderate_missing_cols}")
print(f"  Low missing (<=5%): {low_missing_cols}")

# Column importance analysis
print("\n2. COLUMN IMPORTANCE ANALYSIS:")
critical_cols = ['patient_id', 'age', 'department']  # Critical for analysis
important_cols = ['blood_pressure_systolic', 'heart_rate', 'temperature', 'treatment']  # Important for medical analysis
optional_cols = ['insurance_type', 'length_of_stay', 'total_cost']  # Nice to have

print(f"  Critical columns: {critical_cols}")
print(f"  Important columns: {important_cols}")
print(f"  Optional columns: {optional_cols}")

# Strategy recommendations
print("\n3. STRATEGY RECOMMENDATIONS:")

# For high missing columns
if high_missing_cols:
    print(f"\n  HIGH MISSING COLUMNS ({high_missing_cols}):")
    for col in high_missing_cols:
        if col in critical_cols:
            print(f"    {col}: Consider dropping rows with missing values (critical data)")
        elif col in important_cols:
            print(f"    {col}: Consider filling with domain-specific values")
        else:
            print(f"    {col}: Consider dropping the entire column")

# For moderate missing columns
if moderate_missing_cols:
    print(f"\n  MODERATE MISSING COLUMNS ({moderate_missing_cols}):")
    for col in moderate_missing_cols:
        if col in critical_cols:
            print(f"    {col}: Fill with mode/median based on data type")
        else:
            print(f"    {col}: Fill with appropriate statistical measure")

# For low missing columns
if low_missing_cols:
    print(f"\n  LOW MISSING COLUMNS ({low_missing_cols}):")
    for col in low_missing_cols:
        print(f"    {col}: Fill with mean/median/mode or constant values")

# Final recommendation
print("\n4. FINAL RECOMMENDATION FOR THIS DATASET:")
if len(high_missing_cols) > 0:
    print("  RECOMMENDED: Hybrid approach")
    print("    1. Drop columns with >20% missing (if not critical)")
    print("    2. Fill moderate missing columns with appropriate statistics")
    print("    3. Fill low missing columns with mean/median/mode")
else:
    print("  RECOMMENDED: Fill strategy")
    print("    1. Fill all missing values with appropriate statistical measures")
    print("    2. Preserve all data for maximum information retention")

print("\n5. IMPLEMENTATION DECISION:")
if total_missing / total_cells < 0.1:  # Less than 10% missing
    print("  DECISION: Use combined fill strategy (preserves all data)")
    recommended_df = df_fill_combined.copy()
elif len(high_missing_cols) > 2:
    print("  DECISION: Use selective drop + fill strategy")
    recommended_df = df_drop_specific.copy()
    # Fill remaining missing values
    for col in recommended_df.columns:
        if recommended_df[col].isnull().sum() > 0:
            if recommended_df[col].dtype in ['int64', 'float64']:
                recommended_df[col] = recommended_df[col].fillna(recommended_df[col].median())
            else:
                recommended_df[col] = recommended_df[col].fillna(recommended_df[col].mode()[0])
else:
    print("  DECISION: Use threshold drop + fill strategy")
    recommended_df = df_drop_threshold.copy()
    # Fill remaining missing values
    for col in recommended_df.columns:
        if recommended_df[col].isnull().sum() > 0:
            if recommended_df[col].dtype in ['int64', 'float64']:
                recommended_df[col] = recommended_df[col].fillna(recommended_df[col].median())
            else:
                recommended_df[col] = recommended_df[col].fillna(recommended_df[col].mode()[0])

print(f"\nRecommended final dataset shape: {recommended_df.shape}")
print(f"Recommended missing values: {recommended_df.isnull().sum().sum()}")
print(f"Data retention: {(recommended_df.shape[0] / original_shape[0]) * 100:.1f}%")
print(f"Column retention: {(recommended_df.shape[1] / original_shape[1]) * 100:.1f}%")

## 6. Summary and Conclusions

Comprehensive summary of missing value handling strategies and their impact.

In [ ]:
print("=== COMPREHENSIVE MISSING VALUE HANDLING SUMMARY ===")
print("=" * 60)

# Original dataset summary
print(f"\nORIGINAL DATASET:")
print(f"  Shape: {original_shape}")
print(f"  Missing values: {total_missing}")
print(f"  Missing percentage: {missing_percentage:.2f}%")
print(f"  Columns with missing: {(df.isnull().sum() > 0).sum()}")
print(f"  Rows with missing: {(df.isnull().any(axis=1)).sum()}")

# Strategy effectiveness summary
print(f"\nSTRATEGY EFFECTIVENESS:")
print(f"  Most aggressive (drop any): {df_drop_any.shape[0]} rows retained ({(df_drop_any.shape[0]/original_shape[0])*100:.1f}%)")
print(f"  Most conservative (fill combined): {df_fill_combined.shape[0]} rows retained (100.0%)")
print(f"  Best balance (threshold + fill): {recommended_df.shape[0]} rows retained ({(recommended_df.shape[0]/original_shape[0])*100:.1f}%)")

# Key insights
print(f"\nKEY INSIGHTS:")
print(f"  1. Drop strategies reduce data but ensure complete datasets")
print(f"  2. Fill strategies preserve data but introduce imputed values")
print(f"  3. Combined approaches often provide the best balance")
print(f"  4. Column importance should guide strategy selection")
print(f"  5. Missing percentage influences the appropriate approach")

# Best practices demonstrated
print(f"\nBEST PRACTICES DEMONSTRATED:")
print(f"  1. Always analyze missing patterns before handling")
print(f"  2. Consider data type when choosing fill values")
print(f"  3. Evaluate impact on dataset shape for each strategy")
print(f"  4. Use domain knowledge for critical columns")
print(f"  5. Compare multiple approaches before final decision")

# Final recommendation
print(f"\nFINAL RECOMMENDATION:")
print(f"  Strategy: {'Hybrid drop + fill' if len(high_missing_cols) > 0 else 'Combined fill'}")
print(f"  Final shape: {recommended_df.shape}")
print(f"  Missing values resolved: {recommended_df.isnull().sum().sum() == 0}")
print(f"  Data quality: {'Complete' if recommended_df.isnull().sum().sum() == 0 else 'Partial'}")

print(f"\nHANDLING COMPLETE:")
print(f"  All strategies demonstrated and compared")
print(f"  Decision framework applied")
print(f"  Recommended solution implemented")
print(f"  Dataset ready for analysis")